# Notebook 3 — Train / Validation / Test Split
Split before deep analysis. A time-aware split is preferred here because delivery prediction should mimic future production predictions and reduce temporal leakage. The split keeps chronological order while reporting label balance.

In [ ]:

import os, warnings
from pathlib import Path
import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")

ART = Path("../artifacts")
ART.mkdir(exist_ok=True)
CHARTS = ART / "charts"
CHARTS.mkdir(exist_ok=True)

from sklearn.model_selection import train_test_split

df = pd.read_csv(ART/"02_labeled_table.csv")
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"], errors="coerce")
df = df.sort_values("order_purchase_timestamp").reset_index(drop=True)

print("Date range:", df["order_purchase_timestamp"].min(), "to", df["order_purchase_timestamp"].max())


In [ ]:

# Chronological 70/15/15 split.
n = len(df)
i1 = int(n*0.70)
i2 = int(n*0.85)
train = df.iloc[:i1].copy()
val = df.iloc[i1:i2].copy()
test = df.iloc[i2:].copy()

def balance(x):
    return {
        "rows": len(x),
        "on_time_pct": float((x["late"]==0).mean()),
        "late_pct": float((x["late"]==1).mean()),
        "min_date": x["order_purchase_timestamp"].min(),
        "max_date": x["order_purchase_timestamp"].max()
    }

display(pd.DataFrame({"train":balance(train),"validation":balance(val),"test":balance(test)}).T)


In [ ]:

# Verify chronological separation.
assert train["order_purchase_timestamp"].max() <= val["order_purchase_timestamp"].min()
assert val["order_purchase_timestamp"].max() <= test["order_purchase_timestamp"].min()
print("Chronological split verified.")


In [ ]:

train.to_csv(ART/"03_train.csv", index=False)
val.to_csv(ART/"03_validation.csv", index=False)
test.to_csv(ART/"03_test.csv", index=False)
print("Saved train / validation / test artifacts.")
